In [ ]:
!pip install pandas numpy re json gdown

In [98]:
import pandas as pd
import gdown

file_id = "17H4eZpje6u-Azb2um0sab4je9bAx0Tid"
url = f"https://drive.google.com/uc?id={file_id}"

output = "raw.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df.head()



Downloading...
From: https://drive.google.com/uc?id=17H4eZpje6u-Azb2um0sab4je9bAx0Tid
To: /content/raw.csv
100%|██████████| 2.36M/2.36M [00:00<00:00, 89.8MB/s]


,url,title,text,category_id
0,https://www.ukrinform.ua/rubric-culture/408990...,Івано-Франківський драмтеатр знайшов архівні д...,Про це на пресконференції повідомив гендиректо...,0
1,https://www.ukrinform.ua/rubric-culture/408811...,Премія Олеся Гончара оголосила номінантів,"Як передає Укрінформ, про це повідомило Держми...",0
2,https://www.ukrinform.ua/rubric-culture/408515...,Біографічний фільм «Я граю Роккі» вийде у лист...,"Як передає Укрінформ, про це повідомляє The Ho...",0
3,https://www.ukrinform.ua/rubric-culture/408095...,Netflix підписав угоду на трансляцію фільмів S...,"Про це повідомила Deadline, передає Укрінформ....",0
4,https://www.ukrinform.ua/rubric-culture/408820...,На Венеційській бієнале Україну представить ск...,Про це розповіли учасники Венеційської бієнале...,0


In [99]:
import re
from typing import List, Dict

URL_RE = re.compile(r'https?://[^\s\)\]\}\>,]+')
EMAIL_RE = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')
PHONE_RE = re.compile(r'\+\d[\d\-\(\) ]{7,}\d')
LONG_ID_RE = re.compile(r'\b\d{8,}\b')
HTML_RE = re.compile(r'<[^>]+>')
HTML_RE = re.compile(
    r'<(?!/?(?:URL|EMAIL|PHONE|ID)\b)[^>]+>',
    re.IGNORECASE
)

ABBREVIATIONS = [
    "м.", "вул.", "р.", "рр.", "т.д.", "т.п.", "ім.", "тис.", "дол."
]

PHOTO_CREDIT_RE = re.compile(
    r'(Фото\s*[^.!\n]*|[^.!\n]*?(Укрінформу|Getty|Reuters)[^.!\n]*купити тут)',
    re.IGNORECASE
)

SOCIAL_CREDIT_RE = re.compile(
    r'(Допис|Повідомлення|Публікація)[^(@\n]+?\(@[^)]+\)',
    re.IGNORECASE
)

HOMOGRAPHS = {
    "a": "а", "e": "е", "o": "о", "i": "і", "y": "у", "p": "р", "c": "с", "x": "х"
}

def replace_homographs(word: str) -> str:
    if re.search(r"[а-яіїєґ]", word, re.IGNORECASE):
        for src, dst in HOMOGRAPHS.items():
            word = word.replace(src, dst)
    return word

def remove_photo_credits(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r'Читайте також:', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\.?\s*Більше фото[^.!\n]*', ' ', text, flags=re.IGNORECASE)
    text = PHOTO_CREDIT_RE.sub(" ", text)
    text = SOCIAL_CREDIT_RE.sub(" ", text)

    return text.strip()


def clean_text(text: str) -> str:
    if not text:
        return ""

    text = HTML_RE.sub(" ", text)
    text = remove_photo_credits(text)
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def normalize_text(text: str) -> str:
    text = re.sub(r'[«»“”„]', '"', text)
    text = re.sub(r"[’`´]", "'", text)
    text = re.sub(r'[–—−]', '-', text)
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)

    words = text.split()
    words = [replace_homographs(w) for w in words]
    text = " ".join(words)
    return text.strip()


def mask_pii(text: str) -> str:
    text = URL_RE.sub("<URL>", text)
    text = EMAIL_RE.sub("<EMAIL>", text)
    text = PHONE_RE.sub("<PHONE>", text)
    return text


def sentence_split(text: str) -> List[str]:
    protected = text
    protected = re.sub(r'(\d{1,2}:\d{2})\.', r'\1<DOT_>', text)

    protected = re.sub(
        r'(\d+)\.(?=\s+[A-ZА-ЯІЇЄҐ"])',
        r'\1<DOT>',
        protected
    )
    for abbr in ABBREVIATIONS:
        protected = protected.replace(abbr, abbr.replace(".", "<DOT>"))

    protected = protected.replace('<DOT_>', '.')
    sentences = re.split(r'(?<=[.!?])\s+', protected)
    sentences = [s.replace('<DOT>', '.') for s in sentences]
    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences


def preprocess(text: str) -> Dict:
    clean = clean_text(text)
    norm = normalize_text(clean)
    masked = mask_pii(norm)
    sentences = sentence_split(masked)

    return {
        "clean": masked,
        "sentences": sentences
    }

In [100]:
df["text_clean"] = df["text"].apply(lambda x: preprocess(x)["clean"])
cols = ["title", "text_clean", "category_id"]
OUTPUT_PATH = "processed_v2.csv"
df[cols].to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_PATH)

Saved: processed_v2.csv


In [101]:
samples = df["text"].sample(15)

for t in samples:
    print("RAW:", t)
    print("PROCESSED:", preprocess(t)["clean"])
    print("-"*50)

RAW:  Про це повідомляє Львівська ОВА, передає Укрінформ. «Санкарка зі Львівщини Олена Смага зробила потужний жест солідарності під час виступу на Олімпійських іграх. Після заїзду на одномісних санях вона показала на руці напис: «Пам’ять - це не порушення правил», - йдеться у повідомленні. За словами Олени, це був напис про те, що пам’ять - не політика і що її не можна ні заборонити, ні відняти. Таким чином вона хотіла підтримати Владислава Гераскевича та його команду. «Він неймовірний, це дуже сильний крок», — сказала Олена і висловила жаль з приводу того, що такі акти підтримки часто лишаються непоміченими. Як повідомляв Укрінформ, НОК України закликав МОК дозволити Гераскевичу виступати на Олімпіаді в «Шоломі пам’яті».
PROCESSED: Про це повідомляє Львівська ОВА, передає Укрінформ. "Санкарка зі Львівщини Олена Смага зробила потужний жест солідарності під час виступу на Олімпійських іграх. Після заїзду на одномісних санях вона показала на руці напис: "Пам'ять - це не порушення правил"

In [103]:
empty = (df["text_clean"].str.len() == 0).mean()
short = (df["text_clean"].str.len() < 50).mean()

print(f"Кількість порожніх текстів {empty}")
print(f"Кількість дуже коротких текстів {short}")

Кількість порожніх текстів 0.0
Кількість дуже коротких текстів 0.0


In [104]:
dup_before = df["text"].duplicated().sum()
dup_after = df["text_clean"].duplicated().sum()

print(f"Кількість дублів до: {dup_before} ({dup_before/len(df)*100:.2f} %)")
print(f"Кількість дублів після: {dup_after} ({dup_after/len(df)*100:.2f} %)")

Кількість дублів до: 0 (0.00 %)
Кількість дублів після: 0 (0.00 %)


In [113]:
df['len_chars'] = df['text'].str.len()
df['len_words'] = df['text'].astype(str).apply(lambda x: len(x.split()))
len_char_before = df['len_chars'].mean().round(2)
len_word_before = df['len_words'].mean().round(2)

print(f"Середня довжина текстів у символах (до): {len_char_before}")
print(f"Середня довжина текстів у словах (до): {len_word_before}\n")

df['len_chars'] = df['text_clean'].str.len()
df['len_words'] = df['text_clean'].astype(str).apply(lambda x: len(x.split()))
len_char_after = df['len_chars'].mean().round(2)
len_word_after = df['len_words'].mean().round(2)

print(f"Середня довжина текстів у символах (після): {len_char_after}")
print(f"Середня довжина текстів у словах (після): {len_word_after}")

Середня довжина текстів у символах (до): 1406.56
Середня довжина текстів у словах (до): 192.37

Середня довжина текстів у символах (після): 1396.6
Середня довжина текстів у словах (після): 191.23


In [110]:
df["url_count"] = df["text_clean"].str.count("<URL>")
url_count = df["url_count"].sum()
df["email_count"] = df["text_clean"].str.count("<EMAIL>")
email_count = df["email_count"].sum()
df["phone_count"] = df["text_clean"].str.count("<PHONE>")
phone_count = df["phone_count"].sum()

print(f"Кількість замін (URL/email/phone): {url_count}/{email_count}/{phone_count}")

Кількість замін (URL/email/phone): 4/1/0


In [117]:
import json
summary = {
    "rows": len(df),
    "empty": int(empty),
    "short": int(short),
    "duplicate_rate_before": float(dup_before),
    "duplicate_rate_after": float(dup_after),
    "length_stats_chars_before": float(len_char_before),
    "length_stats_chars_after": float(len_char_after),
    "length_stats_words_before": float(len_word_before),
    "length_stats_words_after": float(len_word_after),
    "mask_counts": {
        "URL": int(url_count),
        "EMAIL": int(email_count),
        "PHONE": int(phone_count),
    }
}

with open("audit_summary_lab2.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

{'rows': 837,
 'empty': 0,
 'short': 0,
 'duplicate_rate_before': 0.0,
 'duplicate_rate_after': 0.0,
 'length_stats_chars_before': 1406.56,
 'length_stats_chars_after': 1396.6,
 'length_stats_words_before': 192.37,
 'length_stats_words_after': 191.23,
 'mask_counts': {'URL': 4, 'EMAIL': 1, 'PHONE': 0}}

Edge cases

In [86]:
import json

cases = []
with open("/content/edge_cases.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        cases.append(json.loads(line))

results = []

for case in cases:
    raw = case["raw_text"]
    processed = preprocess(raw)["clean"]

    results.append({
        "id": case["id"],
        "raw": raw,
        "processed": processed,
        "expected": case["expected_behavior"]
    })

for r in results:
    print(f"\nID {r['id']}")
    print("RAW:", r["raw"])
    print("PROC:", r["processed"])
    print("EXPECT:", r["expected"])


ID 1
RAW: інтернет-магазину BooKMooD (https://bookmoodd.com.ua), власник якого
PROC: інтернет-магазину BooKMooD (<URL>), власник якого
EXPECT: URL замінити на <URL>

ID 2
RAW: Заявки подаються англійською мовою в електронному вигляді на адресу: ubi@ubi.org.ua з темою листа Translate Ukraine 2026.
PROC: Заявки подаються англійською мовою в електронному вигляді на адресу: <EMAIL> з темою листа Translate Ukraine 2026.
EXPECT: EMAIL замінити на <EMAIL>

ID 3
RAW: Віталій Мандзин із чотирма колами штрафу став 28-м (+5.23,5)
PROC: Віталій Мандзин із чотирма колами штрафу став 28-м (+5.23,5)
EXPECT: не розбивати числа з крапкою

ID 4
RAW: Національної премії України ім. Шевченка
PROC: Національної премії України ім. Шевченка
EXPECT: не розбивати речення після 'ім.'

ID 5
RAW: у 2021 році поблизу м. Вишневого Київської області
PROC: у 2021 році поблизу м. Вишневого Київської області
EXPECT: не розбивати речення після 'м.'

ID 6
RAW: У 2026 році на звання лавреата претендують 24 кандидати.
PRO

In [87]:
def test_idempotence(text):
    p1 = preprocess(text)["clean"]
    p2 = preprocess(p1)["clean"]
    return p1 == p2

fails = [c["id"] for c in cases if not test_idempotence(c["raw_text"])]
print("Idempotence failures:", fails)

Idempotence failures: []


In [88]:
def becomes_empty(text):
    if text.strip() == "":
        return False

    processed = preprocess(text)["clean"]
    return processed.strip() == ""

empty_cases = [c["id"] for c in cases if becomes_empty(c["raw_text"])]
print("Unexpected empty after preprocessing:", empty_cases)

Unexpected empty after preprocessing: [11, 12, 20]
